# Player World Cup Experience

## Objective

Create pre-tournament player experience features for each World Cup team. This follows the same leakage-safe method used for coach/manager experience: for a target World Cup, only player appearances before that tournament count as experience.

The output is one row per team-tournament, with squad-level aggregates of prior World Cup experience.

## Inputs

- `worldcup::player_appearances`
- `1.DataCleaning-R/Data/RDS/FullRoster.rds`

## Outputs

- `1.DataCleaning-R/Data/RDS/PlayerExperience.rds`
- `1.DataCleaning-R/Data/CSV/PlayerExperience.csv`

## Packages


In [1]:
library(worldcup)
library(tidyverse)
library(here)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Source Player Appearances

`player_appearances` has one row per player-match appearance. Counting these rows gives matches played at previous World Cups.

In [2]:
player_games <- worldcup::player_appearances

player_games %>%
    group_by(player_id, family_name, given_name) %>%
    summarize(
        matches = n(),
        starts = sum(starter == 1, na.rm = TRUE),
        substitute_appearances = sum(substitute == 1, na.rm = TRUE),
        world_cups = n_distinct(tournament_id),
        .groups = "drop"
    ) %>%
    arrange(desc(matches)) %>%
    print(n = 10)

# A tibble: 6,559 x 7
   player_id family_name     given_name    matches starts substitute_appearances
   <chr>     <chr>           <chr>           <int>  <int>                  <int>
 1 P-14758   "Messi"         Lionel             26     24                      2
 2 P-49502   "Matth\u00e4us" Lothar             25     22                      3
 3 P-25850   "Lilly"         Kristine           24     23                      1
 4 P-27787   "Klose"         Miroslav           24     22                      2
 5 P-62104   "Lloyd"         Carli              24     16                      8
 6 P-89236   "Wambach"       Abby               24     19                      5
 7 P-43222   "Maldini"       Paolo              23     23                      0
 8 P-70442   "Ronaldo"       Cristiano          22     20                      2
 9 P-19610   "Formiga"       not applicab~      21     19                      2
10 P-41187   "Scott"         Jill               21     19                      2
# i 6,

## Team Rosters

Use the roster dataset as the team-tournament structure. Since 2026 rosters are not known yet, this sheet covers tournaments available in the roster data.

In [3]:
full_roster <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullRoster.rds"))

target_tournaments <- full_roster %>%
    distinct(tournament_id) %>%
    arrange(tournament_id) %>%
    pull(tournament_id)

full_roster %>%
    count(tournament_id, team_name, name = "players") %>%
    arrange(tournament_id, team_name) %>%
    print(n = 10)

# A tibble: 128 x 3
   tournament_id team_name players
   <chr>         <chr>       <int>
 1 WC-2010       Algeria        23
 2 WC-2010       Argentina      23
 3 WC-2010       Australia      23
 4 WC-2010       Brazil         23
 5 WC-2010       Cameroon       23
 6 WC-2010       Chile          23
 7 WC-2010       Denmark        23
 8 WC-2010       England        23
 9 WC-2010       France         23
10 WC-2010       Germany        23
# i 118 more rows


## Build Pre-Tournament Player History

For each target World Cup, summarize every player's appearances in earlier World Cups only.

In [4]:
summarize_player_history <- function(target_tournament) {
    player_games %>%
        filter(tournament_id < target_tournament) %>%
        group_by(player_id) %>%
        summarize(
            prior_wc_matches_played = n(),
            prior_wc_starts = sum(starter == 1, na.rm = TRUE),
            prior_wc_substitute_appearances = sum(substitute == 1, na.rm = TRUE),
            prior_wc_group_matches_played = sum(stage_name == "group stage"),
            prior_wc_round_of_16_matches_played = sum(stage_name == "round of 16"),
            prior_wc_quarterfinal_matches_played = sum(stage_name %in% c("quarter-final", "quarter-finals")),
            prior_wc_semifinal_matches_played = sum(stage_name %in% c("semi-final", "semi-finals")),
            prior_wc_third_place_matches_played = sum(stage_name == "third-place match"),
            prior_wc_final_matches_played = sum(stage_name == "final"),
            prior_world_cups_played = n_distinct(tournament_id),
            .groups = "drop"
        ) %>%
        mutate(tournament_id = target_tournament)
}

player_history <- map_dfr(target_tournaments, summarize_player_history)

player_history %>%
    arrange(tournament_id, desc(prior_wc_matches_played)) %>%
    print(n = 10)

# A tibble: 20,221 x 12
   player_id prior_wc_matches_played prior_wc_starts prior_wc_substitute_appea~1
   <chr>                       <int>           <int>                       <int>
 1 P-49502                        25              22                           3
 2 P-25850                        24              23                           1
 3 P-43222                        23              23                           0
 4 P-48883                        21              20                           1
 5 P-80404                        21              21                           0
 6 P-21597                        20              20                           0
 7 P-91718                        20              17                           3
 8 P-59574                        19              15                           4
 9 P-62722                        19              19                           0
10 P-69695                        19              19                           0
# i 

## Join Experience Onto Rosters

Join prior World Cup experience to each player on each roster. Missing values indicate players with no prior World Cup appearances, so they are set to zero.

In [5]:
player_roster_experience <- full_roster %>%
    left_join(player_history, by = c("player_id", "tournament_id")) %>%
    mutate(
        across(
            c(
                prior_wc_matches_played,
                prior_wc_starts,
                prior_wc_substitute_appearances,
                prior_wc_group_matches_played,
                prior_wc_round_of_16_matches_played,
                prior_wc_quarterfinal_matches_played,
                prior_wc_semifinal_matches_played,
                prior_wc_third_place_matches_played,
                prior_wc_final_matches_played,
                prior_world_cups_played
            ),
            ~replace_na(.x, 0L)
        )
    )

player_roster_experience %>%
    arrange(tournament_id, team_name, desc(prior_wc_matches_played)) %>%
    print(n = 10)

# A tibble: 3,039 x 19
   tournament_id team_name team_id team_code player_id family_name  given_name  
   <chr>         <chr>     <chr>   <chr>     <chr>     <chr>        <chr>       
 1 WC-2010       Algeria   T-01    DZA       P-98886   "Gaouaoui"   "Loun\u00e8~
 2 WC-2010       Algeria   T-01    DZA       P-21116   "Bougherra"  "Madjid"    
 3 WC-2010       Algeria   T-01    DZA       P-00957   "Belhadj"    "Nadir"     
 4 WC-2010       Algeria   T-01    DZA       P-77742   "Yahia"      "Antar"     
 5 WC-2010       Algeria   T-01    DZA       P-19453   "Halliche"   "Rafik"     
 6 WC-2010       Algeria   T-01    DZA       P-30221   "Mansouri"   "Yazid"     
 7 WC-2010       Algeria   T-01    DZA       P-70903   "Boudebouz"  "Ryad"      
 8 WC-2010       Algeria   T-01    DZA       P-23484   "Lacen"      "Mehdi"     
 9 WC-2010       Algeria   T-01    DZA       P-22008   "Ghezzal"    "Abdelkader"
10 WC-2010       Algeria   T-01    DZA       P-44112   "Sa\u00effi" "Rafik"     
# i 3

## Aggregate To Team-Tournament Features

Collapse player-level experience into one team-level row per tournament. The totals capture total squad experience, while the counts and averages capture how concentrated or spread out that experience is.

In [6]:
player_experience <- player_roster_experience %>%
    group_by(tournament_id, team_id, team_name, team_code) %>%
    summarize(
        squad_players = n_distinct(player_id),
        players_with_prior_wc = sum(prior_wc_matches_played > 0),
        share_players_with_prior_wc = players_with_prior_wc / squad_players,
        total_prior_wc_matches_played = sum(prior_wc_matches_played),
        mean_prior_wc_matches_played = mean(prior_wc_matches_played),
        max_prior_wc_matches_played = max(prior_wc_matches_played),
        total_prior_wc_starts = sum(prior_wc_starts),
        total_prior_wc_substitute_appearances = sum(prior_wc_substitute_appearances),
        total_prior_wc_group_matches_played = sum(prior_wc_group_matches_played),
        total_prior_wc_round_of_16_matches_played = sum(prior_wc_round_of_16_matches_played),
        total_prior_wc_quarterfinal_matches_played = sum(prior_wc_quarterfinal_matches_played),
        total_prior_wc_semifinal_matches_played = sum(prior_wc_semifinal_matches_played),
        total_prior_wc_third_place_matches_played = sum(prior_wc_third_place_matches_played),
        total_prior_wc_final_matches_played = sum(prior_wc_final_matches_played),
        total_prior_world_cups_played = sum(prior_world_cups_played),
        mean_prior_world_cups_played = mean(prior_world_cups_played),
        max_prior_world_cups_played = max(prior_world_cups_played),
        players_with_prior_wc_final = sum(prior_wc_final_matches_played > 0),
        players_with_multiple_prior_wcs = sum(prior_world_cups_played > 1),
        .groups = "drop"
    )

player_experience %>%
    arrange(tournament_id, team_name) %>%
    print(n = 10)

# A tibble: 128 x 23
   tournament_id team_id team_name team_code squad_players players_with_prior_wc
   <chr>         <chr>   <chr>     <chr>             <int>                 <int>
 1 WC-2010       T-01    Algeria   DZA                  23                     0
 2 WC-2010       T-03    Argentina ARG                  23                     8
 3 WC-2010       T-04    Australia AUS                  23                    12
 4 WC-2010       T-09    Brazil    BRA                  23                     7
 5 WC-2010       T-11    Cameroon  CMR                  23                     3
 6 WC-2010       T-13    Chile     CHL                  23                     0
 7 WC-2010       T-22    Denmark   DNK                  23                     6
 8 WC-2010       T-28    England   ENG                  23                    11
 9 WC-2010       T-30    France    FRA                  23                     8
10 WC-2010       T-31    Germany   DEU                  23                     7
# i 118

## Quick Checks

Confirm there is one row per team-tournament and inspect the strongest experience totals.

In [7]:
player_experience %>%
    count(tournament_id, name = "teams")

player_experience %>%
    summarize(
        rows = n(),
        team_tournament_rows = n_distinct(tournament_id, team_name),
        min_squad_players = min(squad_players),
        max_squad_players = max(squad_players),
        teams_with_prior_wc_players = sum(players_with_prior_wc > 0)
    )

player_experience %>%
    select(tournament_id, team_name, squad_players, players_with_prior_wc, total_prior_wc_matches_played, total_prior_wc_final_matches_played) %>%
    arrange(desc(total_prior_wc_matches_played)) %>%
    print(n = 10)

tournament_id,teams
<chr>,<int>
WC-2010,32
WC-2014,32
WC-2018,32
WC-2022,32


rows,team_tournament_rows,min_squad_players,max_squad_players,teams_with_prior_wc_players
<int>,<int>,<int>,<int>,<int>
128,128,23,26,106


# A tibble: 128 x 6
   tournament_id team_name squad_players players_with_prior_wc
   <chr>         <chr>             <int>                 <int>
 1 WC-2014       Spain                23                    14
 2 WC-2014       Germany              23                    11
 3 WC-2022       Belgium              26                    14
 4 WC-2022       Uruguay              26                    13
 5 WC-2018       Germany              23                     9
 6 WC-2022       France               26                    10
 7 WC-2018       Argentina            23                     9
 8 WC-2014       Uruguay              23                    14
 9 WC-2010       Italy                23                     9
10 WC-2018       Mexico               23                    13
# i 118 more rows
# i 2 more variables: total_prior_wc_matches_played <int>,
#   total_prior_wc_final_matches_played <int>


## Save

Save RDS version.

In [8]:
saveRDS(player_experience, here("1.DataCleaning-R", "Data", "RDS", "PlayerExperience.rds"))